# Genex RAG v1 — Real HPO + PubMed Demo

This notebook upgrades the earlier Ask Genex prototype from a tiny toy corpus to a **real HPO-backed RAG v1**.

## What this notebook does
- loads **real HPO processed tables**
  - `condition.parquet`
  - `feature.parquet`
  - `condition_feature.parquet`
- optionally enriches HPO terms from raw `hp.json`
- loads the **CDC milestone table**
- asks for a **child profile**
  - name
  - age in months
  - known condition
  - parent concerns
- pulls **top PubMed papers** for the child context + question
- retrieves from:
  - HPO term docs
  - HPO-derived condition profile docs
  - CDC milestone docs
  - PubMed paper docs
- compares multiple answering modes / models
- computes a small comparison table of practical metrics
- generates a **Next Doctor Visit Prep** summary

## Intended use
This notebook is for:
- Genex product prototyping
- professor / advisor demos
- understanding the architecture line by line
- iterating toward a stronger RAG system before app integration

## Important note
This is still **RAG v1**:
- retrieval is simple and transparent
- scoring and metrics are lightweight
- it is designed to be easy to inspect and edit

## Recommended placement

Put this notebook in:

`Genex/notebooks/mvp/`

It will try to find:
- `hpo/data_proc/condition.parquet`
- `hpo/data_proc/feature.parquet`
- `hpo/data_proc/condition_feature.parquet`
- `hpo/data_raw/hpo/hp.json` (optional)
- `milestone-cdc-table.xlsx`

If the auto-detection fails, you can manually pass paths in the load cells.

In [ ]:
# Install once in this notebook kernel if needed
%pip install pandas pyarrow openpyxl scikit-learn python-dotenv openai requests ipython

In [ ]:
# ------------------------------------------------------------
# 1. Imports + setup
# ------------------------------------------------------------
import os
import re
import json
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display, Markdown

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()
warnings.filterwarnings("ignore", category=UserWarning)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("PROJECT_ROOT:", PROJECT_ROOT)

if OpenAI is None:
    print("Warning: openai package not available. LLM comparisons will be unavailable.")
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY not found. Extractive mode will still work.")

In [ ]:
# ------------------------------------------------------------
# 2. File finders
# ------------------------------------------------------------
def find_first_match(filename: str, preferred_subpaths: Optional[List[str]] = None) -> Path:
    search_roots = [Path.cwd(), PROJECT_ROOT, PROJECT_ROOT.parent]
    candidates = []

    if preferred_subpaths:
        for root in search_roots:
            for sub in preferred_subpaths:
                p = root / sub / filename
                if p.exists():
                    return p.resolve()

    for root in search_roots:
        direct = root / filename
        if direct.exists():
            return direct.resolve()

    for root in search_roots:
        if root.exists():
            candidates.extend(root.rglob(filename))

    if candidates:
        return candidates[0].resolve()

    raise FileNotFoundError(f"Could not find {filename}")

CONDITION_PATH = find_first_match("condition.parquet", preferred_subpaths=["hpo/data_proc", "data_proc", "hpo/data/proc"])
FEATURE_PATH = find_first_match("feature.parquet", preferred_subpaths=["hpo/data_proc", "data_proc", "hpo/data/proc"])
CONDITION_FEATURE_PATH = find_first_match("condition_feature.parquet", preferred_subpaths=["hpo/data_proc", "data_proc", "hpo/data/proc"])

try:
    HP_JSON_PATH = find_first_match("hp.json", preferred_subpaths=["hpo/data_raw/hpo", "data_raw/hpo", "raw/hpo"])
except Exception:
    HP_JSON_PATH = None

CDC_PATH = find_first_match("milestone-cdc-table.xlsx", preferred_subpaths=["", "data", "docs", "notebooks"])

print("condition.parquet ->", CONDITION_PATH)
print("feature.parquet ->", FEATURE_PATH)
print("condition_feature.parquet ->", CONDITION_FEATURE_PATH)
print("hp.json ->", HP_JSON_PATH)
print("CDC milestone table ->", CDC_PATH)

In [ ]:
# ------------------------------------------------------------
# 3. Load HPO processed tables + CDC
# ------------------------------------------------------------
condition_df = pd.read_parquet(CONDITION_PATH)
feature_df = pd.read_parquet(FEATURE_PATH)
condition_feature_df = pd.read_parquet(CONDITION_FEATURE_PATH)

cdc_df = pd.read_excel(CDC_PATH)
cdc_df.columns = [str(c).strip().lower() for c in cdc_df.columns]
cdc_df = cdc_df.rename(columns={"category ": "category", "milestone ": "milestone"})
cdc_df["category"] = cdc_df["category"].astype(str).str.strip().str.lower()
cdc_df["milestone"] = cdc_df["milestone"].astype(str).str.strip()
cdc_df["months"] = pd.to_numeric(cdc_df["months"], errors="coerce")
cdc_df = cdc_df.dropna(subset=["months", "category", "milestone"]).copy()
cdc_df["months"] = cdc_df["months"].astype(int)

ALIAS_TO_CATEGORY = {
    "movement and physical": "movement_and_physical",
    "physical": "movement_and_physical",
    "motor": "movement_and_physical",
    "social and emotional": "social_and_emotional",
    "social and emotial": "social_and_emotional",
    "social": "social_and_emotional",
    "language and communication": "language_and_communication",
    "speech": "language_and_communication",
    "language": "language_and_communication",
    "speech and language": "language_and_communication",
    "cognitive": "cognitive",
    "adaptive": "cognitive",
}

DOMAIN_DISPLAY = {
    "movement_and_physical": "Movement / Physical",
    "social_and_emotional": "Social / Emotional",
    "language_and_communication": "Language / Communication",
    "cognitive": "Cognitive / Adaptive",
}

cdc_df["category_key"] = cdc_df["category"].map(lambda x: ALIAS_TO_CATEGORY.get(x, x.replace(" ", "_")))

print("condition_df:", condition_df.shape)
print("feature_df:", feature_df.shape)
print("condition_feature_df:", condition_feature_df.shape)
print("cdc_df:", cdc_df.shape)
display(condition_df.head())
display(feature_df.head())
display(condition_feature_df.head())
display(cdc_df.head())

In [ ]:
# ------------------------------------------------------------
# 4. Optional hp.json enrichment (definitions + synonyms)
# ------------------------------------------------------------
def parse_hp_json(path: Optional[Path]) -> pd.DataFrame:
    if path is None or not Path(path).exists():
        return pd.DataFrame(columns=["feature_id", "label_json", "definition", "synonyms"])

    with open(path, "r", encoding="utf-8") as f:
        hp = json.load(f)

    rows = []
    graphs = hp.get("graphs", [])
    if not graphs:
        return pd.DataFrame(columns=["feature_id", "label_json", "definition", "synonyms"])

    nodes = graphs[0].get("nodes", [])
    for node in nodes:
        node_id = node.get("id", "")
        if not str(node_id).startswith("HP:"):
            continue

        label = node.get("lbl", "")
        meta = node.get("meta", {}) or {}
        definition = ""
        if isinstance(meta.get("definition"), dict):
            definition = meta.get("definition", {}).get("val", "") or ""

        synonyms = []
        for syn in meta.get("synonyms", []) or []:
            if isinstance(syn, dict) and syn.get("val"):
                synonyms.append(syn["val"])

        rows.append({
            "feature_id": node_id,
            "label_json": label,
            "definition": definition,
            "synonyms": "; ".join(sorted(set(synonyms))) if synonyms else "",
        })

    return pd.DataFrame(rows)

hp_meta_df = parse_hp_json(HP_JSON_PATH)
print("hp_meta_df:", hp_meta_df.shape)
display(hp_meta_df.head())

In [ ]:
# ------------------------------------------------------------
# 5. Build HPO term docs
# ------------------------------------------------------------
def build_hpo_term_docs(feature_df: pd.DataFrame, hp_meta_df: pd.DataFrame) -> pd.DataFrame:
    term_df = feature_df.copy()

    if not hp_meta_df.empty:
        term_df = term_df.merge(hp_meta_df, on="feature_id", how="left")
    else:
        term_df["label_json"] = None
        term_df["definition"] = None
        term_df["synonyms"] = None

    term_df["best_label"] = term_df["label_json"].fillna(term_df["label"]).fillna(term_df["feature_id"])
    term_df["definition"] = term_df["definition"].fillna("")
    term_df["synonyms"] = term_df["synonyms"].fillna("")

    rows = []
    for _, row in term_df.iterrows():
        pieces = [f"HPO term {row['feature_id']}", f"Label: {row['best_label']}"]
        if row["definition"]:
            pieces.append(f"Definition: {row['definition']}")
        if row["synonyms"]:
            pieces.append(f"Synonyms: {row['synonyms']}")
        if pd.notna(row.get("ic")):
            pieces.append(f"Information content: {round(float(row['ic']), 4)}")

        rows.append({
            "doc_id": f"hpo_term_{row['feature_id']}",
            "source": "hpo_term",
            "title": f"HPO {row['feature_id']} {row['best_label']}",
            "text": " ".join(pieces),
            "feature_id": row["feature_id"],
            "label": row["best_label"],
        })

    return pd.DataFrame(rows)

hpo_term_docs_df = build_hpo_term_docs(feature_df, hp_meta_df)
print("hpo_term_docs_df:", hpo_term_docs_df.shape)
display(hpo_term_docs_df.head())

In [ ]:
# ------------------------------------------------------------
# 6. Build HPO-derived condition profile docs
# ------------------------------------------------------------
def build_condition_profile_docs(
    condition_df: pd.DataFrame,
    condition_feature_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    hp_meta_df: pd.DataFrame,
    top_terms_per_condition: int = 20,
) -> pd.DataFrame:
    feat = feature_df.copy()
    if not hp_meta_df.empty:
        feat = feat.merge(hp_meta_df[["feature_id", "label_json"]], on="feature_id", how="left")
        feat["best_label"] = feat["label_json"].fillna(feat["label"]).fillna(feat["feature_id"])
    else:
        feat["best_label"] = feat["label"].fillna(feat["feature_id"])

    merged = (
        condition_feature_df
        .merge(feat[["feature_id", "best_label", "ic"]], on="feature_id", how="left")
        .merge(condition_df[["condition_id", "name"]], on="condition_id", how="left")
    )

    merged["name"] = merged["name"].fillna(merged["condition_id"])
    merged["best_label"] = merged["best_label"].fillna(merged["feature_id"])
    merged["ic"] = merged["ic"].fillna(0.0)
    merged["weight"] = merged["weight"].fillna(1.0)
    merged["score"] = merged["weight"] * merged["ic"]

    rows = []
    for condition_id, sub in merged.groupby("condition_id", sort=False):
        sub = sub.sort_values(["score", "weight", "best_label"], ascending=[False, False, True]).copy()
        top_terms = sub.head(top_terms_per_condition)
        condition_name = str(top_terms["name"].iloc[0]) if len(top_terms) else str(condition_id)
        phenotypes = top_terms["best_label"].astype(str).tolist()

        rows.append({
            "doc_id": f"condition_profile_{condition_id}",
            "source": "condition_profile",
            "title": f"Condition profile: {condition_name}",
            "text": (
                f"Condition {condition_name} ({condition_id}). "
                f"Top phenotype terms: {'; '.join(phenotypes)}. "
                f"These phenotype terms are derived from HPO condition-to-feature associations."
            ),
            "condition_id": condition_id,
            "condition_name": condition_name,
            "top_phenotypes": phenotypes,
        })

    return pd.DataFrame(rows)

condition_docs_df = build_condition_profile_docs(condition_df, condition_feature_df, feature_df, hp_meta_df, 20)
print("condition_docs_df:", condition_docs_df.shape)
display(condition_docs_df.head())

In [ ]:
# ------------------------------------------------------------
# 7. Build CDC milestone docs
# ------------------------------------------------------------
def build_cdc_docs(cdc_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    grouped = cdc_df.groupby(["category_key", "months"], dropna=False)

    for (category_key, months), subset in grouped:
        category_display = DOMAIN_DISPLAY.get(category_key, category_key)
        milestone_lines = "\n".join([f"- {m}" for m in subset["milestone"].tolist()])

        rows.append({
            "doc_id": f"cdc_{category_key}_{months}",
            "source": "cdc",
            "title": f"CDC {category_display} {months} months",
            "text": f"CDC milestone reference for {category_display} at {months} months. Milestones:\n{milestone_lines}",
            "category_key": category_key,
            "months": int(months),
        })

    return pd.DataFrame(rows)

cdc_docs_df = build_cdc_docs(cdc_df)
print("cdc_docs_df:", cdc_docs_df.shape)
display(cdc_docs_df.head())

In [ ]:
# ------------------------------------------------------------
# 8. Build and fit the static retriever
# ------------------------------------------------------------
def build_static_corpus(hpo_term_docs_df, condition_docs_df, cdc_docs_df) -> pd.DataFrame:
    cols = ["doc_id", "source", "title", "text"]
    df = pd.concat([
        hpo_term_docs_df[cols].copy(),
        condition_docs_df[cols].copy(),
        cdc_docs_df[cols].copy(),
    ], ignore_index=True)
    df["text_for_retrieval"] = df["title"].fillna("") + "\n" + df["text"].fillna("")
    return df

static_corpus_df = build_static_corpus(hpo_term_docs_df, condition_docs_df, cdc_docs_df)
print("static_corpus_df:", static_corpus_df.shape)

static_vectorizer = TfidfVectorizer(lowercase=True, stop_words="english", ngram_range=(1, 2), min_df=1)
static_doc_matrix = static_vectorizer.fit_transform(static_corpus_df["text_for_retrieval"])

def retrieve_static_docs(query: str, top_k: int = 12, source_filter: Optional[List[str]] = None) -> pd.DataFrame:
    df = static_corpus_df.copy()

    if source_filter:
        df = df[df["source"].isin(source_filter)].copy()
        if df.empty:
            return df
        idx = df.index.tolist()
        qvec = static_vectorizer.transform([query])
        sims = cosine_similarity(qvec, static_doc_matrix[idx]).flatten()
        df["score"] = sims
        return df.sort_values("score", ascending=False).head(top_k)

    qvec = static_vectorizer.transform([query])
    sims = cosine_similarity(qvec, static_doc_matrix).flatten()
    df["score"] = sims
    return df.sort_values("score", ascending=False).head(top_k)

display(retrieve_static_docs("speech delay 48 month child language milestones", top_k=8)[["doc_id", "source", "title", "score"]])

In [ ]:
# ------------------------------------------------------------
# 9. Child context intake
# ------------------------------------------------------------
def collect_child_context() -> Dict[str, Any]:
    print("Enter child context for Ask Genex RAG v1")
    name = input("Child name: ").strip()
    age_months = int(input("Child age in months: ").strip())
    condition = input("Known condition / diagnosis (type 'none' if none): ").strip()
    concern = input("Main parent concerns: ").strip()

    return {
        "child_name": name,
        "age_months": age_months,
        "condition": condition,
        "concern": concern,
    }

# Uncomment to use live input:
# child_ctx = collect_child_context()

# Example fallback:
child_ctx = {
    "child_name": "Ari",
    "age_months": 48,
    "condition": "speech delay",
    "concern": "speech delay, developmental delay, difficulty expressing needs",
}

child_ctx

In [ ]:
# ------------------------------------------------------------
# 10. PubMed helpers (official NCBI E-utilities)
# ------------------------------------------------------------
EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def build_pubmed_query(child_ctx: Dict[str, Any], user_query: str) -> str:
    parts = []

    condition = str(child_ctx.get("condition", "")).strip()
    concern = str(child_ctx.get("concern", "")).strip()
    age_months = child_ctx.get("age_months")

    if condition and condition.lower() != "none":
        parts.append(condition)
    if concern:
        parts.append(concern)
    if user_query:
        parts.append(user_query)

    if age_months is not None:
        years = round(age_months / 12.0, 1)
        parts.append(f"pediatric child age {years} years")

    return " AND ".join([p for p in parts if p])

def pubmed_esearch(query: str, retmax: int = 5) -> List[str]:
    params = {"db": "pubmed", "term": query, "retmode": "json", "sort": "relevance", "retmax": retmax}
    api_key = os.environ.get("NCBI_API_KEY")
    if api_key:
        params["api_key"] = api_key

    r = requests.get(f"{EUTILS_BASE}/esearch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.json().get("esearchresult", {}).get("idlist", [])

def pubmed_efetch_xml(pmids: List[str]) -> str:
    if not pmids:
        return ""

    params = {"db": "pubmed", "id": ",".join(pmids), "retmode": "xml"}
    api_key = os.environ.get("NCBI_API_KEY")
    if api_key:
        params["api_key"] = api_key

    r = requests.get(f"{EUTILS_BASE}/efetch.fcgi", params=params, timeout=60)
    r.raise_for_status()
    return r.text

def parse_pubmed_xml(xml_text: str) -> pd.DataFrame:
    if not xml_text.strip():
        return pd.DataFrame(columns=["pmid", "title", "abstract", "journal", "year"])

    root = ET.fromstring(xml_text)
    rows = []

    for article in root.findall(".//PubmedArticle"):
        pmid = article.findtext(".//PMID", default="")
        title = article.findtext(".//ArticleTitle", default="") or ""

        abstract_parts = []
        for abs_node in article.findall(".//Abstract/AbstractText"):
            label = abs_node.attrib.get("Label", "")
            txt = "".join(abs_node.itertext()).strip()
            if label and txt:
                abstract_parts.append(f"{label}: {txt}")
            elif txt:
                abstract_parts.append(txt)
        abstract = " ".join(abstract_parts).strip()

        journal = article.findtext(".//Journal/Title", default="")
        year = article.findtext(".//PubDate/Year", default="") or article.findtext(".//ArticleDate/Year", default="")

        rows.append({
            "pmid": pmid,
            "title": title.strip(),
            "abstract": abstract,
            "journal": journal,
            "year": year,
        })

    return pd.DataFrame(rows)

def fetch_pubmed_papers(query: str, retmax: int = 5) -> pd.DataFrame:
    try:
        pmids = pubmed_esearch(query=query, retmax=retmax)
        xml_text = pubmed_efetch_xml(pmids)
        return parse_pubmed_xml(xml_text)
    except Exception as e:
        print("PubMed fetch warning:", e)
        return pd.DataFrame(columns=["pmid", "title", "abstract", "journal", "year"])

def build_pubmed_docs(papers_df: pd.DataFrame) -> pd.DataFrame:
    if papers_df.empty:
        return pd.DataFrame(columns=["doc_id", "source", "title", "text", "pmid"])

    rows = []
    for _, row in papers_df.iterrows():
        rows.append({
            "doc_id": f"pubmed_{row['pmid']}",
            "source": "pubmed",
            "title": f"PubMed PMID {row['pmid']}: {row['title']}",
            "text": (
                f"PMID: {row['pmid']}. Journal: {row['journal']}. Year: {row['year']}. "
                f"Title: {row['title']}. Abstract: {row['abstract']}"
            ),
            "pmid": row["pmid"],
        })
    return pd.DataFrame(rows)

test_pubmed_query = build_pubmed_query(child_ctx, "speech delay and developmental concerns")
print("PubMed query:", test_pubmed_query)
display(fetch_pubmed_papers(test_pubmed_query, retmax=3))

In [ ]:
# ------------------------------------------------------------
# 11. Query routing and context assembly
# ------------------------------------------------------------
def detect_query_type(query: str) -> str:
    q = query.lower()
    if any(t in q for t in ["doctor", "appointment", "visit", "what should i ask", "what should i bring"]):
        return "doctor_prep"
    if any(t in q for t in ["paper", "papers", "study", "studies", "research", "article", "evidence"]):
        return "literature_qna"
    if any(t in q for t in ["hpo", "phenotype", "symptom", "feature", "traits", "which term", "what term"]):
        return "phenotype_qna"
    if any(t in q for t in ["milestone", "month", "months", "speech", "motor", "social", "language", "adaptive", "what should i track"]):
        return "milestone_qna"
    return "general_qna"

def source_filter_for_query_type(query_type: str) -> List[str]:
    if query_type == "milestone_qna":
        return ["cdc", "condition_profile", "hpo_term"]
    if query_type == "phenotype_qna":
        return ["hpo_term", "condition_profile"]
    if query_type == "literature_qna":
        return ["condition_profile", "hpo_term", "cdc"]
    if query_type == "doctor_prep":
        return ["condition_profile", "hpo_term", "cdc"]
    return ["condition_profile", "hpo_term", "cdc"]

def assemble_context(child_ctx: Dict[str, Any], user_query: str, top_k_static: int = 10, top_k_papers: int = 5) -> Dict[str, Any]:
    query_type = detect_query_type(user_query)
    source_filter = source_filter_for_query_type(query_type)

    child_query = (
        f"child age {child_ctx['age_months']} months; "
        f"condition {child_ctx['condition']}; "
        f"concern {child_ctx['concern']}; "
        f"user question {user_query}"
    )

    static_results = retrieve_static_docs(child_query, top_k=top_k_static, source_filter=source_filter).copy()

    pubmed_query = build_pubmed_query(child_ctx, user_query)
    papers_df = fetch_pubmed_papers(pubmed_query, retmax=top_k_papers)
    pubmed_docs_df = build_pubmed_docs(papers_df)

    if not pubmed_docs_df.empty:
        pubmed_docs_df["score"] = [1.0 - (i * 0.05) for i in range(len(pubmed_docs_df))]

    context_docs = pd.concat(
        [
            static_results[["doc_id", "source", "title", "text", "score"]].copy(),
            pubmed_docs_df[["doc_id", "source", "title", "text", "score"]].copy() if not pubmed_docs_df.empty else pd.DataFrame(columns=["doc_id", "source", "title", "text", "score"])
        ],
        ignore_index=True,
    )

    return {
        "query_type": query_type,
        "source_filter": source_filter,
        "pubmed_query": pubmed_query,
        "static_results": static_results,
        "papers_df": papers_df,
        "pubmed_docs_df": pubmed_docs_df,
        "context_docs": context_docs,
    }

In [ ]:
# ------------------------------------------------------------
# 12. Formatting helpers
# ------------------------------------------------------------
def format_context_docs(context_docs: pd.DataFrame) -> str:
    blocks = []
    for _, row in context_docs.iterrows():
        blocks.append(
            f"[doc_id: {row['doc_id']}]\n"
            f"source: {row['source']}\n"
            f"title: {row['title']}\n"
            f"score: {row['score']}\n"
            f"text: {row['text']}"
        )
    return "\n\n".join(blocks)

def simple_citations(context_docs: pd.DataFrame, max_items: int = 8) -> List[Dict[str, Any]]:
    return [
        {"doc_id": row["doc_id"], "source": row["source"], "title": row["title"]}
        for _, row in context_docs.head(max_items).iterrows()
    ]

def count_unique_sources(context_docs: pd.DataFrame) -> int:
    return int(context_docs["source"].nunique()) if not context_docs.empty else 0

In [ ]:
# ------------------------------------------------------------
# 13. Answer generators
# ------------------------------------------------------------
def answer_extractively(child_ctx: Dict[str, Any], user_query: str, context_docs: pd.DataFrame) -> Dict[str, Any]:
    if context_docs.empty:
        return {
            "model_name": "extractive_baseline",
            "answer": "I could not find enough supporting documents for this question.",
            "citations": [],
            "follow_up_questions": [
                "Can you clarify the main concern?",
                "Is there a known diagnosis?",
                "Which developmental area matters most right now?",
            ],
            "safety_flags": ["insufficient_evidence", "non_diagnostic"],
        }

    top_docs = context_docs.head(4).copy()
    lines = [
        f"Genex found {len(context_docs)} relevant evidence items for {child_ctx['child_name']}.",
        f"This child is {child_ctx['age_months']} months old, with condition '{child_ctx['condition']}' and parent concern '{child_ctx['concern']}'.",
        f"The question appears to be about: {user_query}",
    ]
    if "pubmed" in set(top_docs["source"]):
        lines.append("Relevant PubMed papers were also retrieved for this question.")
    lines.append("Top retrieved evidence includes:")
    for _, row in top_docs.iterrows():
        lines.append(f"- [{row['source']}] {row['title']}")

    return {
        "model_name": "extractive_baseline",
        "answer": " ".join(lines),
        "citations": simple_citations(context_docs),
        "follow_up_questions": [
            "Would you like Genex to focus on milestones, phenotype terms, or papers?",
            "Would you like a next doctor visit prep summary?",
        ],
        "safety_flags": ["extractive_mode", "non_diagnostic"],
    }

def answer_with_llm(child_ctx: Dict[str, Any], user_query: str, context_docs: pd.DataFrame, model_name: str) -> Dict[str, Any]:
    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "model_name": model_name,
            "answer": f"Model {model_name} unavailable because OPENAI_API_KEY is missing.",
            "citations": simple_citations(context_docs),
            "follow_up_questions": [],
            "safety_flags": ["llm_unavailable"],
        }

    client = OpenAI()
    context_text = format_context_docs(context_docs)

    prompt = f"""
You are Ask Genex, a retrieval-grounded pediatric developmental and condition support assistant.

Rules:
1. Use only the retrieved context below.
2. Stay non-diagnostic.
3. Use parent-friendly language.
4. Say clearly if support is limited or uncertain.
5. Do not invent facts not supported by the retrieved documents.
6. Return strict JSON only.

Child profile:
- Name: {child_ctx['child_name']}
- Age: {child_ctx['age_months']} months
- Known condition: {child_ctx['condition']}
- Parent concern: {child_ctx['concern']}

User question:
{user_query}

Retrieved context:
{context_text}

Required JSON:
{{
  "answer": "...",
  "citations": [
    {{"doc_id": "...", "title": "..."}}
  ],
  "follow_up_questions": ["..."],
  "safety_flags": ["..."]
}}
""".strip()

    resp = client.chat.completions.create(
        model=model_name,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You return strict JSON only and stay non-diagnostic."},
            {"role": "user", "content": prompt},
        ],
    )

    parsed = json.loads(resp.choices[0].message.content)
    return {
        "model_name": model_name,
        "answer": parsed.get("answer", ""),
        "citations": parsed.get("citations", []),
        "follow_up_questions": parsed.get("follow_up_questions", []),
        "safety_flags": parsed.get("safety_flags", []),
    }

In [ ]:
# ------------------------------------------------------------
# 14. Comparison metrics + model runner
# ------------------------------------------------------------
def evaluate_answer_bundle(bundle: Dict[str, Any], context_docs: pd.DataFrame, papers_df: pd.DataFrame) -> Dict[str, Any]:
    answer = bundle.get("answer", "") or ""
    citations = bundle.get("citations", []) or []
    follow_up_questions = bundle.get("follow_up_questions", []) or []
    safety_flags = bundle.get("safety_flags", []) or []

    uncertainty_terms = ["may", "might", "could", "uncertain", "limited", "not enough", "insufficient"]
    mentions_uncertainty = any(t in answer.lower() for t in uncertainty_terms)
    mentions_doctor = ("doctor" in answer.lower()) or ("clinician" in answer.lower())

    return {
        "model_name": bundle.get("model_name", ""),
        "answer_words": len(answer.split()),
        "answer_chars": len(answer),
        "citation_count": len(citations),
        "follow_up_count": len(follow_up_questions),
        "safety_flag_count": len(safety_flags),
        "context_doc_count": len(context_docs),
        "unique_context_sources": count_unique_sources(context_docs),
        "pubmed_paper_count": len(papers_df),
        "mentions_uncertainty": mentions_uncertainty,
        "mentions_doctor": mentions_doctor,
    }

def compare_models(child_ctx: Dict[str, Any], user_query: str, model_names: Optional[List[str]] = None, top_k_static: int = 10, top_k_papers: int = 5) -> Dict[str, Any]:
    if model_names is None:
        model_names = ["extractive_baseline", "gpt-4o-mini", "gpt-4.1-mini"]

    assembled = assemble_context(child_ctx, user_query, top_k_static=top_k_static, top_k_papers=top_k_papers)
    context_docs = assembled["context_docs"]
    papers_df = assembled["papers_df"]

    bundles = []
    metrics_rows = []

    for model_name in model_names:
        if model_name == "extractive_baseline":
            bundle = answer_extractively(child_ctx, user_query, context_docs)
        else:
            bundle = answer_with_llm(child_ctx, user_query, context_docs, model_name=model_name)

        bundles.append(bundle)
        metrics_rows.append(evaluate_answer_bundle(bundle, context_docs, papers_df))

    metrics_df = pd.DataFrame(metrics_rows)

    return {
        "child_ctx": child_ctx,
        "user_query": user_query,
        "query_type": assembled["query_type"],
        "pubmed_query": assembled["pubmed_query"],
        "papers_df": papers_df,
        "context_docs": context_docs,
        "bundles": bundles,
        "metrics_df": metrics_df,
    }

In [ ]:
# ------------------------------------------------------------
# 15. Next doctor visit prep
# ------------------------------------------------------------
def prepare_next_doctor_visit(child_ctx: Dict[str, Any], user_query: Optional[str] = None, model_name: str = "gpt-4o-mini", top_k_static: int = 10, top_k_papers: int = 5) -> Dict[str, Any]:
    prep_query = user_query or "Prepare the next doctor appointment summary for this child."
    assembled = assemble_context(child_ctx, prep_query, top_k_static=top_k_static, top_k_papers=top_k_papers)
    context_docs = assembled["context_docs"]

    if OpenAI is None or not os.environ.get("OPENAI_API_KEY"):
        return {
            "summary": f"Doctor visit prep for {child_ctx['child_name']}",
            "top_concerns": [child_ctx["concern"]],
            "questions_to_ask": [
                "What developmental areas should we prioritize next?",
                "What should we track at home before the next visit?",
                "Are there referrals or therapies we should discuss?",
            ],
            "records_to_bring": [
                "Recent developmental observations",
                "Therapy reports if available",
                "Questions about milestones, communication, motor, social, or adaptive skills",
            ],
            "citations": simple_citations(context_docs),
            "safety_flags": ["extractive_doctor_prep", "non_diagnostic"],
        }

    client = OpenAI()
    context_text = format_context_docs(context_docs)

    prompt = f"""
You are Genex Doctor Visit Prep, a retrieval-grounded visit preparation assistant.

Rules:
1. Use only the retrieved context below.
2. Stay non-diagnostic.
3. Write for a parent preparing for a pediatric / developmental / genetics visit.
4. Keep the output structured, practical, and concise.
5. Return strict JSON only.

Child profile:
- Name: {child_ctx['child_name']}
- Age: {child_ctx['age_months']} months
- Known condition: {child_ctx['condition']}
- Parent concern: {child_ctx['concern']}

Retrieved context:
{context_text}

Required JSON:
{{
  "summary": "...",
  "top_concerns": ["..."],
  "questions_to_ask": ["..."],
  "records_to_bring": ["..."],
  "citations": [
    {{"doc_id": "...", "title": "..."}}
  ],
  "safety_flags": ["..."]
}}
""".strip()

    resp = client.chat.completions.create(
        model=model_name,
        temperature=0.2,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content": "You return strict JSON only and stay non-diagnostic."},
            {"role": "user", "content": prompt},
        ],
    )

    return json.loads(resp.choices[0].message.content)

In [ ]:
# ------------------------------------------------------------
# 16. Run one full Ask Genex comparison
# ------------------------------------------------------------
# You can edit these interactively:
user_query = "What should I track at home for this child's speech and development, and what are the most relevant papers?"

# Uncomment these two lines to use live input:
# child_ctx = collect_child_context()
# user_query = input("Parent question for Ask Genex: ").strip()

MODEL_LIST = ["extractive_baseline", "gpt-4o-mini", "gpt-4.1-mini"]

comparison = compare_models(
    child_ctx=child_ctx,
    user_query=user_query,
    model_names=MODEL_LIST,
    top_k_static=10,
    top_k_papers=5,
)

print("Child context:", comparison["child_ctx"])
print("Query type:", comparison["query_type"])
print("PubMed query:", comparison["pubmed_query"])

print("\nTop papers:")
display(comparison["papers_df"])

print("\nRetrieved context docs:")
display(comparison["context_docs"][["doc_id", "source", "title", "score"]].head(15))

print("\nModel comparison metrics:")
display(comparison["metrics_df"])

In [ ]:
# ------------------------------------------------------------
# 17. Print answers side by side
# ------------------------------------------------------------
for bundle in comparison["bundles"]:
    print("\n" + "=" * 100)
    print("MODEL:", bundle["model_name"])
    print("=" * 100)
    print("ANSWER:\n")
    print(bundle["answer"])
    print("\nCITATIONS:")
    for c in bundle.get("citations", []):
        print("-", c)
    print("\nFOLLOW-UP QUESTIONS:")
    for q in bundle.get("follow_up_questions", []):
        print("-", q)
    print("\nSAFETY FLAGS:", bundle.get("safety_flags", []))

In [ ]:
# ------------------------------------------------------------
# 18. Generate next doctor visit prep
# ------------------------------------------------------------
doctor_prep = prepare_next_doctor_visit(
    child_ctx=child_ctx,
    user_query="Prepare the next doctor visit summary for this child.",
    model_name="gpt-4o-mini",
    top_k_static=10,
    top_k_papers=5,
)

doctor_prep

## How to extend this notebook next

### Strong next steps
1. Add better parent-language ↔ HPO synonym mapping  
2. Use the HPO matrix later for phenotype-to-condition ranking or reranking  
3. Cache PubMed results  
4. Add a small gold evaluation set with advisor-reviewed expected answers  
5. Compare:
   - extractive baseline
   - grounded RAG with one model
   - grounded RAG with another model
   - later, fine-tuned behavior models

### Product directions
- Ask Genex chat tab
- Next doctor appointment prep tab
- parent-friendly evidence cards
- structured JSON for app integration
- later: incorporate child progress tracked in the app

In [ ]:
# ------------------------------------------------------------
# 19. Optional: save outputs
# ------------------------------------------------------------
from datetime import datetime

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "rag_v1_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

save_bundle = {
    "child_ctx": comparison["child_ctx"],
    "user_query": comparison["user_query"],
    "query_type": comparison["query_type"],
    "pubmed_query": comparison["pubmed_query"],
    "papers": comparison["papers_df"].to_dict(orient="records"),
    "context_docs": comparison["context_docs"].to_dict(orient="records"),
    "answers": comparison["bundles"],
    "metrics": comparison["metrics_df"].to_dict(orient="records"),
    "doctor_prep": doctor_prep,
}

save_path = OUTPUT_DIR / f"genex_rag_v1_{RUN_STAMP}.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_bundle, f, indent=2)

print("Saved:", save_path)